# TO-Agents — run the demo

A group of AI agents turns a written description of a structural problem into a
validated setup, runs a 3D topology optimization, looks at the rendered result
with a vision model, proposes revisions, and scores the candidates.

**To start:** add an API key to Secrets (the 🔑 icon on the left), then
**Runtime → Run all**. It takes about four minutes and prints a link.

| key | cost | where |
|---|---|---|
| `GEMINI_API_KEY` | free, no card | https://aistudio.google.com/apikey |
| `OPENROUTER_API_KEY` | ~$0.002 per run | https://openrouter.ai/keys |

Either one works. Give the notebook access to it with the toggle beside it.


In [ ]:
#@title  { display-mode: "form" }
# Everything happens in quickstart.py so this stays one cell.
import os, subprocess, sys

REPO_DIR = '/content/to-agents-lite'
REPO_URL = 'https://github.com/bellastewart/to-agents-lite.git'

# --- key, from Colab Secrets --------------------------------------------------
_found = []
try:
    from google.colab import userdata
    for _canonical, _aliases in {
        'GEMINI_API_KEY':     ['GEMINI_API_KEY', 'GEMINI', 'Gemini', 'GOOGLE_API_KEY'],
        'OPENROUTER_API_KEY': ['OPENROUTER_API_KEY', 'OPENROUTER', 'Openrouter',
                               'OpenRouter', 'openrouter'],
    }.items():
        for _a in _aliases:
            try:
                _v = userdata.get(_a)
            except Exception:
                _v = None
            if _v:
                os.environ[_canonical] = _v.strip()
                _found.append(_canonical)
                break
except Exception:
    pass

if not _found:
    import getpass
    _v = getpass.getpass('Paste a Gemini or OpenRouter API key (then press Enter): ').strip()
    if not _v:
        raise SystemExit(
            'No key given. Add GEMINI_API_KEY or OPENROUTER_API_KEY to Secrets '
            '(the key icon on the left) and run this cell again.')
    os.environ['OPENROUTER_API_KEY' if _v.startswith('sk-or') else 'GEMINI_API_KEY'] = _v

# --- code ---------------------------------------------------------------------
if os.path.isdir(REPO_DIR + '/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], capture_output=True)
else:
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], capture_output=True)

# --- everything else ----------------------------------------------------------
# Stream the child's output line by line rather than subprocess.run().
#
# A subprocess inherits the kernel's REAL stdout (file descriptor 1), but
# IPython captures cell output by swapping sys.stdout at the Python level. So
# subprocess.run() prints nothing here -- the progress lines and the URL go to
# the kernel log where nobody sees them, and the cell shows only the returned
# CompletedProcess repr. Piping and re-printing puts it back in the cell, and
# has the side benefit of appearing live instead of all at the end.
_p = subprocess.Popen([sys.executable, 'quickstart.py'], cwd=REPO_DIR,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1)
for _line in _p.stdout:
    print(_line, end='')
_p.wait()

# Trailing semicolon-free expressions get echoed by Jupyter; keep the cell quiet.
if _p.returncode != 0:
    print('\nSetup did not finish. The lines above say why.')
None
